In [1]:
import pandas as pd
import numpy as np

KeyboardInterrupt: 

In [ ]:
df_sq = pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\squads.csv')
df_bts=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\batting_stats.csv')
df_bs=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\bowling_stats.csv')
df_pt=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\points_table.csv')
df_mt=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\matches.csv')
 #added prev years data from 2008-onwards

print(df_sq.isnull().sum())
print(df_bts.isnull().sum())
print(df_bs.isnull().sum())
print(df_pt.isnull().sum())
print(df_mt.isnull().sum())

#drop the row in which match was abandoned 
df_mt=df_mt[df_mt['match_result']=='completed']
print(df_mt.isnull().sum())
df_mt = df_mt[df_mt['match_result'] == 'completed']
df_mt['date'] = pd.to_datetime(df_mt['date'], format='%B %d, %Y')   

team_no          0
team_name        0
player           0
nationality      0
role             0
designation    247
dtype: int64
position       0
batsman        0
team           0
matches        0
innings        0
runs           0
impact         0
average        0
strike_rate    0
not_outs       0
high_score     0
balls_faced    0
hundreds       0
fifties        0
ducks          0
fours          0
sixes          0
dtype: int64
position        0
bowler          0
team            0
matches         0
innings         0
wickets         0
economy         0
impact          0
bbi             0
overs           0
balls           0
dot_balls       0
maiden_overs    0
runs            0
avg             0
4wh             0
5wh             0
dtype: int64
position     0
team         0
matches      0
wins         0
defeats      0
ties         0
abandoned    0
points       0
nrr          0
dtype: int64
match_id                0
date                    0
venue                   0
team1                   0


In [ ]:


df_bs['dotball_ratio'] = df_bs['dot_balls'] / df_bs['balls']
#dropping all the features not needed by us for our model
df_sq=df_sq[['team_name','player']]
df_bts=df_bts[['batsman','average','impact','strike_rate']]
df_bs=df_bs[['bowler','avg','impact','economy','dotball_ratio']]
df_mt=df_mt[['date','team1','team2','match_winner']]
df_pt=df_pt[['team','points','nrr']]


#features

#headtohead, teamrating, batting strength , bowling stregth, previous match performances

df_pred=pd.DataFrame(columns=['Team Name','Team Rating','Batting Strength','Bowling Strength','Star Presence'])
#made a new df and added the 
df_pred['Team Name']=df_pt['team']
print(df_pred['Team Name'])


#headtohead of two teams dataframe 


0                   Punjab Kings
1    Royal Challengers Bengaluru
2            Sunrisers Hyderabad
3               Rajasthan Royals
4                 Gujarat Titans
5            Chennai Super Kings
6                 Delhi Capitals
7          Kolkata Knight Riders
8                 Mumbai Indians
9           Lucknow Super Giants
Name: Team Name, dtype: object


In [ ]:
# ============================================================
# Load & clean player stats, compute Batting Strength / Bowling Strength
# ============================================================
df_players = pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\squad_with_stats_FINAL.csv')

# Fix typos in the source file so team names match the rest of the pipeline
team_name_fixes = {
    'Gujrat Titans': 'Gujarat Titans',
    'Kolkata Night Riders': 'Kolkata Knight Riders',
}
df_players['team_name'] = df_players['team_name'].replace(team_name_fixes)

# Sanity check: every name here should now also appear in df_pred['Team Name']
print(sorted(df_players['team_name'].unique()))
print(sorted(df_pred['Team Name'].unique()))

stat_cols = ['t20_batting_avg','t20_bowling_avg','t20_batting_sr','t20_bowling_sr','t20_bowling_econ',
             'ipl_batting_avg','ipl_bowling_avg','ipl_batting_sr','ipl_bowling_sr','ipl_bowling_econ']

# Step 1: convert to numeric, treating '-' as missing
for col in stat_cols:
    df_players[col] = pd.to_numeric(df_players[col].replace('-', pd.NA), errors='coerce')

# Step 2: prefer IPL record; fall back to international (t20) record if IPL is missing
df_players['eff_batting_avg'] = df_players['ipl_batting_avg'].fillna(df_players['t20_batting_avg'])
df_players['eff_batting_sr']  = df_players['ipl_batting_sr'].fillna(df_players['t20_batting_sr'])
df_players['eff_bowling_avg'] = df_players['ipl_bowling_avg'].fillna(df_players['t20_bowling_avg'])
df_players['eff_bowling_econ']= df_players['ipl_bowling_econ'].fillna(df_players['t20_bowling_econ'])

# Step 3: an all-zero pair isn't a real score (0.0 average forever is impossible) — treat as missing
df_players.loc[(df_players['eff_batting_avg'] == 0) & (df_players['eff_batting_sr'] == 0),
               ['eff_batting_avg','eff_batting_sr']] = None
df_players.loc[(df_players['eff_bowling_avg'] == 0) & (df_players['eff_bowling_econ'] == 0),
               ['eff_bowling_avg','eff_bowling_econ']] = None

# Step 4: identify players with genuinely no record anywhere (first season / no intl history)
no_batting_data = df_players['eff_batting_avg'].isna() & df_players['eff_batting_sr'].isna()
no_bowling_data = df_players['eff_bowling_avg'].isna() & df_players['eff_bowling_econ'].isna()

print("Excluded from batting calc (no record found):")
print(df_players[no_batting_data & df_players['role'].isin(['Batter','All Rounder','Wicket Keeper'])]['player'].tolist())
print("\nExcluded from bowling calc (no record found):")
print(df_players[no_bowling_data & df_players['role'].isin(['Bowler','All Rounder'])]['player'].tolist())

# Step 5: Batting Strength — Batter / All Rounder / Wicket Keeper
# (assumption: wicket keepers included since they always bat too)
batting_eligible = df_players[df_players['role'].isin(['Batter','All Rounder','Wicket Keeper']) & ~no_batting_data]
team_batting_strength = batting_eligible.groupby('team_name')[['eff_batting_avg','eff_batting_sr']].mean()
team_batting_strength['Batting Strength'] = (
    team_batting_strength['eff_batting_avg'] * 0.5 + team_batting_strength['eff_batting_sr'] * 0.5
)

# Step 6: Bowling Strength — Bowler / All Rounder
bowling_eligible = df_players[df_players['role'].isin(['Bowler','All Rounder']) & ~no_bowling_data]
team_bowling_strength = bowling_eligible.groupby('team_name')[['eff_bowling_avg','eff_bowling_econ']].mean()
team_bowling_strength['Bowling Strength'] = (
    (1 / team_bowling_strength['eff_bowling_avg']) * 0.5 + (1 / team_bowling_strength['eff_bowling_econ']) * 0.5
)

# Step 7: merge into df_pred
df_pred['Batting Strength'] = df_pred['Team Name'].map(team_batting_strength['Batting Strength'])
df_pred['Bowling Strength'] = df_pred['Team Name'].map(team_bowling_strength['Bowling Strength'])

print("\n", df_pred)
print("\nMissing values:\n", df_pred.isnull().sum())

['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
Excluded from batting calc (no record found):
['Rohit Sharma', 'Danish Malewar', 'Atharva Ankolekar', 'Mayank Rawat', 'Kartik Sharma', 'Ayush Matre', 'Aman Khan', 'Shivam Dubey', 'Zak Foulkes', 'Ramakrishna Ghosh', 'Prashant Veer', 'Finn Allen', 'Sarthak Ranjan', 'Tejasvi Dahiya', 'Daksh Kamra', 'Devdutt Padikal', 'Phil Salt', 'Jordan Cox', 'Mangesh Yadav', 'Vihaan Malhotra', 'Kanishk Chouhan', 'Nishant Sindhu', 'Rashid Khan', 'Aman Rao', 'Shubham Dubey', 'Ravi Singh', 'Lhuan-dre Pretorius', 'Ravichandran Smaran', 'Salil Arora', 'Harsh Dubey', 'Krains Fuletra'

In [ ]:
#check if a top performer from either bowling rating or batting rating is present in the squad od one team 

#is the union of the star performers of bnatting and bowling of the entire tournament
star_players=set(df_bts['batsman']).union(set(df_bs['bowler']))


df_sq['is_star']=df_sq['player'].isin(star_players)

df_pred['Star Presence']=df_pred['Team Name'].map(df_sq.groupby('team_name')['is_star'].sum())


df_pred['Star Presence']=df_pred['Star Presence'].fillna(0)

#completed my star presence feature and now moving on to team rating and squad strength features

team_rating = (df_pt['points']*0.2 + df_pt['nrr']).set_axis(df_pt['team'])
df_pred['Team Rating']=df_pred['Team Name'].map(team_rating)
print(df_pred)

#completed my team rating feature and now moving on to squad strength features

                     Team Name  Team Rating  Batting Strength  \
0                 Punjab Kings        3.933         74.322000   
1  Royal Challengers Bengaluru        4.319         91.964375   
2          Sunrisers Hyderabad        2.815         85.346000   
3             Rajasthan Royals        2.602         80.681111   
4               Gujarat Titans        1.125         66.368333   
5          Chennai Super Kings        1.079         85.047778   
6               Delhi Capitals        0.140         79.346667   
7        Kolkata Knight Riders        1.751         82.674583   
8               Mumbai Indians        0.064         75.737692   
9         Lucknow Super Giants       -0.306         76.606000   

   Bowling Strength  Star Presence  
0          0.066410            3.0  
1          0.067084            4.0  
2          0.068630            4.0  
3          0.070251            4.0  
4          0.072533            0.0  
5          0.067794            3.0  
6          0.075191      

In [ ]:
# Load the extra historical matches
df_mt_prev = pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\matches_prev_years.csv')
df_mt_prev['date'] = pd.to_datetime(df_mt_prev['date'], format='%Y-%m-%d')   # ADD THIS LINE, right after loading df_mt_prev
# Extract a clean 4-digit year from season, handling both "2015" and "2007/08" formats
df_mt_prev['season_year'] = df_mt_prev['season'].str.extract(r'(\d{4})').astype(int)

# Keep only the last 7 seasons
df_mt_prev = df_mt_prev[df_mt_prev['season_year'] >= 2018]

# Map old/retired franchise names to their current names
team_rename_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}
df_mt_prev['team1'] = df_mt_prev['team1'].replace(team_rename_map)
df_mt_prev['team2'] = df_mt_prev['team2'].replace(team_rename_map)
df_mt_prev['winner'] = df_mt_prev['winner'].replace(team_rename_map)

# Align columns to match df_mt's structure, then combine
df_mt_prev = df_mt_prev[['date', 'team1', 'team2', 'winner']].rename(columns={'winner': 'match_winner'})
df_mt_combined = pd.concat(
    [df_mt_prev, df_mt[['date', 'team1', 'team2', 'match_winner']]],
    ignore_index=True
)
df_mt_combined['date'] = pd.to_datetime(df_mt_combined['date'])
df_mt_combined = df_mt_combined.sort_values('date').reset_index(drop=True)

print(df_mt_combined.shape)

print(df_mt_combined['season_year'].min() if 'season_year' in df_mt_combined else "combined ok")
df_headtohead=pd.DataFrame(columns=['Team1','Team2',' Winner'])


(497, 4)
combined ok


In [ ]:

# Head-to-head feature + match features, using FULL team names 
# consistently across current season and historical data

team_name_map = {
    'RCB': 'Royal Challengers Bengaluru', 'SRH': 'Sunrisers Hyderabad',
    'MI': 'Mumbai Indians', 'KKR': 'Kolkata Knight Riders',
    'PBKS': 'Punjab Kings', 'RR': 'Rajasthan Royals',
    'GT': 'Gujarat Titans', 'LSG': 'Lucknow Super Giants',
    'DC': 'Delhi Capitals', 'CSK': 'Chennai Super Kings'
}

#Map this season's abbreviations to full names
df_mt['team1_full'] = df_mt['team1'].map(team_name_map)
df_mt['team2_full'] = df_mt['team2'].map(team_name_map)
df_mt['match_winner_full'] = df_mt['match_winner'].map(team_name_map)

print("Unmapped team1/team2 check (should all be 0):")
print(df_mt[['team1_full', 'team2_full']].isnull().sum())

# Merge team1's and team2's pre-computed features (rating, star presence etc.)
df_matches_features = df_mt.merge(
    df_pred, left_on='team1_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team1_rating', 'Batting Strength': 'team1_batting_strength',
    'Bowling Strength': 'team1_bowling_strength', 'Star Presence': 'team1_star_presence'
}).drop(columns=['Team Name'])

df_matches_features = df_matches_features.merge(
    df_pred, left_on='team2_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team2_rating', 'Batting Strength': 'team2_batting_strength',
    'Bowling Strength': 'team2_bowling_strength', 'Star Presence': 'team2_star_presence'
}).drop(columns=['Team Name'])

#Build the target variable
df_matches_features['team1_won'] = (
    df_matches_features['match_winner'] == df_matches_features['team1']
).astype(int)

df_matches_features = df_matches_features.sort_values('date').reset_index(drop=True)



#Build a FULL combined history (historical + current season), all using full team names
df_mt_current_full = df_mt[['date', 'team1_full', 'team2_full', 'match_winner_full']].rename(
    columns={'team1_full': 'team1', 'team2_full': 'team2', 'match_winner_full': 'match_winner'}
)

df_mt_combined = pd.concat([df_mt_prev, df_mt_current_full], ignore_index=True)
df_mt_combined = df_mt_combined.sort_values('date').reset_index(drop=True)

print("\nCombined history shape:", df_mt_combined.shape)

# Head-to-head lookup, filtered by DATE (not row index), using full names
def get_head_to_head(team1, team2, match_date, history_df):
    h2h_matches = history_df[
        (history_df['date'] < match_date) &
        (((history_df['team1'] == team1) & (history_df['team2'] == team2)) |
         ((history_df['team1'] == team2) & (history_df['team2'] == team1)))
    ]
    total = len(h2h_matches)
    if total == 0:
        return 0.5
    team1_wins = (h2h_matches['match_winner'] == team1).sum()
    return team1_wins / total

df_matches_features['team1_h2h_win_rate'] = df_matches_features.apply(
    lambda row: get_head_to_head(row['team1_full'], row['team2_full'], row['date'], df_mt_combined),
    axis=1
)

#NOW it's safe to drop the helper columns, since h2h calc is done
df_matches_features = df_matches_features.drop(columns=['team1_full', 'team2_full', 'match_winner_full'])

print(df_matches_features.head())
print("\n", df_matches_features[['date', 'team1', 'team2', 'match_winner', 'team1_h2h_win_rate']].head(15))
print("\nh2h value distribution:\n", df_matches_features['team1_h2h_win_rate'].value_counts())

Unmapped team1/team2 check (should all be 0):
team1_full    0
team2_full    0
dtype: int64

Combined history shape: (497, 4)
        date team1 team2 match_winner  team1_rating  team1_batting_strength  \
0 2026-03-28   RCB   SRH          RCB         4.319               91.964375   
1 2026-03-29    MI   KKR           MI         0.064               75.737692   
2 2026-03-30    RR   CSK           RR         2.602               80.681111   
3 2026-03-31  PBKS    GT         PBKS         3.933               74.322000   
4 2026-04-01   LSG    DC           DC        -0.306               76.606000   

   team1_bowling_strength  team1_star_presence  team2_rating  \
0                0.067084                  4.0         2.815   
1                0.072941                  0.0         1.751   
2                0.070251                  4.0         1.079   
3                0.066410                  3.0         1.125   
4                0.070101                  2.0         0.140   

   team2_battin

In [ ]:
df_current_season = df_mt.merge(
    df_pred, left_on='team1_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team1_rating', 'Batting Strength': 'team1_batting_strength',
    'Bowling Strength': 'team1_bowling_strength', 'Star Presence': 'team1_star_presence'
}).drop(columns=['Team Name'])

df_current_season = df_current_season.merge(
    df_pred, left_on='team2_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team2_rating', 'Batting Strength': 'team2_batting_strength',
    'Bowling Strength': 'team2_bowling_strength', 'Star Presence': 'team2_star_presence'
}).drop(columns=['Team Name'])

df_current_season['team1_won'] = (
    df_current_season['match_winner'] == df_current_season['team1']
).astype(int)

df_current_season = df_current_season.sort_values('date').reset_index(drop=True)

print("Current season shape:", df_current_season.shape)
print(df_current_season.isnull().sum())

Current season shape: (38, 16)
date                      0
team1                     0
team2                     0
match_winner              0
team1_full                0
team2_full                0
match_winner_full         0
team1_rating              0
team1_batting_strength    0
team1_bowling_strength    0
team1_star_presence       0
team2_rating              0
team2_batting_strength    0
team2_bowling_strength    0
team2_star_presence       0
team1_won                 0
dtype: int64


In [ ]:
#grt recent rolling form of the team 
def get_recent_form(team, match_date, history_df, n=10):
    team_matches = history_df[
        ((history_df['team1'] == team) | (history_df['team2'] == team)) &
        (history_df['date'] < match_date)
    ].tail(n)
    if len(team_matches) == 0:
        return 0.5
    wins = (team_matches['match_winner'] == team).sum()
    return wins / len(team_matches)



df_current_season['team1_h2h_win_rate'] = df_current_season.apply(
    lambda row: get_head_to_head(row['team1_full'], row['team2_full'], row['date'], df_mt_combined),
    axis=1
)

df_current_season['team1_form'] = df_current_season.apply(
    lambda row: get_recent_form(row['team1_full'], row['date'], df_mt_combined), axis=1
)
df_current_season['team2_form'] = df_current_season.apply(
    lambda row: get_recent_form(row['team2_full'], row['date'], df_mt_combined), axis=1
)

print(df_current_season[['date', 'team1', 'team2', 'team1_h2h_win_rate', 'team1_form', 'team2_form']])

         date team1 team2  team1_h2h_win_rate  team1_form  team2_form
0  2026-03-28   RCB   SRH            0.500000         0.6         0.5
1  2026-03-29    MI   KKR            0.538462         0.3         0.8
2  2026-03-30    RR   CSK            0.583333         0.5         0.5
3  2026-03-31  PBKS    GT            0.400000         0.3         0.4
4  2026-04-01   LSG    DC            0.600000         0.4         0.6
5  2026-04-02   KKR   SRH            0.687500         0.7         0.4
6  2026-04-03   CSK  PBKS            0.461538         0.4         0.4
7  2026-04-04    GT    RR            0.833333         0.3         0.5
8  2026-04-04    DC    MI            0.466667         0.7         0.3
9  2026-04-05   SRH   LSG            0.250000         0.5         0.4
10 2026-04-05   RCB   CSK            0.307692         0.7         0.3
11 2026-04-07    RR    MI            0.615385         0.5         0.3
12 2026-04-08    DC    GT            0.600000         0.7         0.3
13 2026-04-09   KKR 

In [ ]:
feature_columns_combined = [
    'team1_rating', 'team2_rating',
    'team1_batting_strength', 'team2_batting_strength',
    'team1_bowling_strength', 'team2_bowling_strength',
    'team1_star_presence', 'team2_star_presence',
    'team1_h2h_win_rate',
    'team1_form', 'team2_form'
]

X = df_current_season[feature_columns_combined]
y = df_current_season['team1_won']

print("Feature matrix shape:", X.shape)
print("Missing values:\n", X.isnull().sum())

Feature matrix shape: (38, 11)
Missing values:
 team1_rating              0
team2_rating              0
team1_batting_strength    0
team2_batting_strength    0
team1_bowling_strength    0
team2_bowling_strength    0
team1_star_presence       0
team2_star_presence       0
team1_h2h_win_rate        0
team1_form                0
team2_form                0
dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

split_index = int(len(df_current_season) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

importance_df = pd.DataFrame({
    'feature': feature_columns_combined,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
print("\nFeature importance:\n", importance_df)

results_df = X_test.copy()
results_df['actual_team1_won'] = y_test.values
results_df['predicted_team1_won'] = y_pred
results_df['team1_win_probability'] = y_pred_proba[:, 1]
results_df['team2_win_probability'] = y_pred_proba[:, 0]

print("\nTest set predictions with probabilities:\n",
      results_df[['actual_team1_won', 'predicted_team1_won', 'team1_win_probability', 'team2_win_probability']])


Train size: 30, Test size: 8

Accuracy: 0.5

Classification Report:
               precision    recall  f1-score   support

           0       0.60      0.60      0.60         5
           1       0.33      0.33      0.33         3

    accuracy                           0.50         8
   macro avg       0.47      0.47      0.47         8
weighted avg       0.50      0.50      0.50         8


Confusion Matrix:
 [[3 2]
 [2 1]]

Feature importance:
                    feature  coefficient
0             team1_rating     0.632612
6      team1_star_presence     0.270013
3   team2_batting_strength     0.152787
1             team2_rating     0.120937
5   team2_bowling_strength     0.004041
4   team1_bowling_strength     0.000813
8       team1_h2h_win_rate    -0.001871
2   team1_batting_strength    -0.023205
10              team2_form    -0.043717
9               team1_form    -0.134260
7      team2_star_presence    -0.487962

Test set predictions with probabilities:
     actual_team1_won  p

In [ ]:
#cell 11
#cross validation and baseline comparison 
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print("CV accuracy per fold:", scores)
print("Mean CV accuracy:", scores.mean(), "±", scores.std())

baseline = y.value_counts(normalize=True).max()
print("Naive baseline (always guess majority class):", baseline)

CV accuracy per fold: [0.625      0.5        0.75       0.57142857 0.28571429]
Mean CV accuracy: 0.5464285714285714 ± 0.1538618516324144
Naive baseline (always guess majority class): 0.5526315789473685


In [ ]:
#cell 12
#feature selection 
from sklearn.linear_model import LogisticRegressionCV

model_l1 = LogisticRegressionCV(
    Cs=10, cv=cv, penalty='l1', solver='liblinear', max_iter=1000
)
model_l1.fit(StandardScaler().fit_transform(X), y)

importance_l1 = pd.DataFrame({
    'feature': feature_columns_combined,
    'coefficient': model_l1.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print(importance_l1)

                   feature  coefficient
0             team1_rating     0.560805
4   team1_bowling_strength    -0.198056
5   team2_bowling_strength     0.078193
1             team2_rating     0.000000
2   team1_batting_strength     0.000000
3   team2_batting_strength     0.000000
6      team1_star_presence     0.000000
7      team2_star_presence     0.000000
8       team1_h2h_win_rate     0.000000
9               team1_form     0.000000
10              team2_form     0.000000


In [ ]:
# Trimmed feature set — only what L1 kept
feature_columns_trimmed = ['team1_rating', 'team1_bowling_strength', 'team2_bowling_strength']

X_trimmed = df_current_season[feature_columns_trimmed]

pipeline_trimmed = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

scores_trimmed = cross_val_score(pipeline_trimmed, X_trimmed, y, cv=cv, scoring='accuracy')
print("Trimmed CV accuracy per fold:", scores_trimmed)
print("Trimmed mean CV accuracy:", scores_trimmed.mean(), "±", scores_trimmed.std())
print("Naive baseline:", baseline)

Trimmed CV accuracy per fold: [1.         0.625      0.5        0.71428571 0.71428571]
Trimmed mean CV accuracy: 0.7107142857142857 ± 0.16459598031147019
Naive baseline: 0.5526315789473685


In [ ]:
#cell 14
# Simplest possible model — just the one clearly dominant feature
X_single = df_current_season[['team1_rating']]

scores_single = cross_val_score(pipeline_trimmed, X_single, y, cv=cv, scoring='accuracy')
print("Single-feature CV accuracy per fold:", scores_single)
print("Single-feature mean CV accuracy:", scores_single.mean(), "±", scores_single.std())

Single-feature CV accuracy per fold: [0.75       0.75       0.75       0.42857143 0.57142857]
Single-feature mean CV accuracy: 0.65 ± 0.1305404777321219


In [ ]:
# Final model — trained on the full dataset, using the validated feature set
final_model = LogisticRegression(max_iter=1000)
final_scaler = StandardScaler()

X_final = df_current_season[feature_columns_trimmed]
X_final_scaled = final_scaler.fit_transform(X_final)

final_model.fit(X_final_scaled, y)


print(f"Features used: {feature_columns_trimmed}")
print(f"\nMean CV Accuracy: {scores_trimmed.mean():.3f} ± {scores_trimmed.std():.3f}")
print(f"Per-fold accuracy: {scores_trimmed}")
print(f"Naive baseline (always guess majority class): {baseline:.3f}")

print("\nOverall Accuracy (out-of-fold):", accuracy_score(y, y_pred_cv))

print("\nClassification Report (out-of-fold):\n", classification_report(y, y_pred_cv))

print("Confusion Matrix (out-of-fold):\n", confusion_matrix(y, y_pred_cv))

train_predictions = final_model.predict(X_final_scaled)
train_probabilities = final_model.predict_proba(X_final_scaled)

fit_summary = df_current_season[['date', 'team1', 'team2', 'match_winner']].copy()
fit_summary['team1_won'] = y.values
fit_summary['predicted_team1_won'] = train_predictions
fit_summary['team1_win_probability'] = train_probabilities[:, 1]

print("\nHow the final model fits its own training data (NOT a real accuracy measure):")
print(fit_summary.head(15))

Features used: ['team1_rating', 'team1_bowling_strength', 'team2_bowling_strength']

Mean CV Accuracy: 0.711 ± 0.165
Per-fold accuracy: [1.         0.625      0.5        0.71428571 0.71428571]
Naive baseline (always guess majority class): 0.553

Overall Accuracy (out-of-fold): 0.7105263157894737

Classification Report (out-of-fold):
               precision    recall  f1-score   support

           0       0.69      0.65      0.67        17
           1       0.73      0.76      0.74        21

    accuracy                           0.71        38
   macro avg       0.71      0.70      0.71        38
weighted avg       0.71      0.71      0.71        38

Confusion Matrix (out-of-fold):
 [[11  6]
 [ 5 16]]

How the final model fits its own training data (NOT a real accuracy measure):
         date team1 team2 match_winner  team1_won  predicted_team1_won  \
0  2026-03-28   RCB   SRH          RCB          1                    1   
1  2026-03-29    MI   KKR           MI          1         

: 